# CFG Reduction Using Leiden Community Detection Algorithm

This notebook is a copy of `wip.ipynb` with an extra community detection step to collapse the graph into fewer nodes that are more appropriately sized for LLM summarization. 


### 1. Extract CFG from IDA Database
This step requires `idapro` 9.0+ and a valid license.


In [ ]:
import sys
import os
from src.extract_cfg import extract_cfg_from_db

# Replace with the path to your IDA database (.idb or .i64)
db_path = "/Users/mark/windows_share/test/reorder_and_pad.exe.i64"
json_path = "../data/parsed_xref_graph.json"

if os.path.exists(db_path):
    json_path = extract_cfg_from_db(db_path, output_path=json_path)
else:
    print(f"Database {db_path} not found. Skipping extraction.")


### 2. Leiden Community Detection Algorithm

This algorithm dtects communities by iteratively refining partitions. So far I've tried using the Constant Potts Model (CPM) partition type, which optimizes:

$$Q_{CPM} = \sum_c \left[ \sum_{i,j \in c} w_{ij} - \gamma |n_c| \right]$$

where $w_{ij}$ is an edge weight, $|n_c|$ is community size, and $\gamma$ (resolution) controls granularity. ([paper](https://arxiv.org/pdf/1810.08473)).

Since the original CFG does not have weighted edges, and the algorithm does consider these weights, artificial weights have been added for different edge types: inter-function calls = 0.1, intra-function edges = 2.0, and non-call edges = 1.0. 

It is probably worth trying out different combinations of edge weights, resolution parameters, and parition types.

_NetworkX does not have a built-in implementation of Leiden, so most of this function is converting to an igraph graph, running igraph's implementation, and then converting back to networkx._

In [ ]:
#!/usr/bin/env python3
import json
import networkx as nx
from networkx.algorithms import community

import leidenalg
import igraph


def _assign_edge_weights(G):
    for u, v, data in G.edges(data=True):
        edge_type = data.get('type', 'unknown')
        
        if edge_type == 'inter-function':
            data['weight'] = 0.1
        elif edge_type == 'intra-function':
            data['weight'] = 2.0
        elif edge_type == 'non-call':
            data['weight'] = 1.0
        else:
            data['weight'] = 1.0


def collapse_leiden(G, resolution=0.05, weighted=True, partition_type='CPM'):
    # Map partition type strings to leidenalg classes
    partition_classes = {
        'CPM': leidenalg.CPMVertexPartition,
        'RB': leidenalg.RBConfigurationVertexPartition,
        'Modularity': leidenalg.ModularityVertexPartition,
        'Significance': leidenalg.SignificanceVertexPartition,
    }
    
    if partition_type not in partition_classes:
        raise ValueError(
            f"Unknown partition_type '{partition_type}'. "
            f"Choose from: {list(partition_classes.keys())}"
        )
    
    partition_class = partition_classes[partition_type]
    
    # Assign edge weights based on type for better community detection
    if weighted:
        _assign_edge_weights(G)
    
    # Create igraph from NetworkX directed graph with node attribute mapping
    nodes = list(G.nodes())
    node_to_idx = {node: idx for idx, node in enumerate(nodes)}
    
    # Build edges with remapped indices
    edges = [(node_to_idx[u], node_to_idx[v]) for u, v in G.edges()]
    
    ig = igraph.Graph(len(nodes), edges, directed=True)
    ig.vs['node_id'] = nodes
    
    # Add edge weights to igraph
    edge_weights = []
    for u, v in G.edges():
        edge_data = G.get_edge_data(u, v) or {}
        weight = edge_data.get('weight', 1.0)
        edge_weights.append(weight)
    ig.es['weight'] = edge_weights
    
    # Apply Leiden algorithm with specified partition type
    partition = leidenalg.find_partition(
        ig,
        partition_class,
        weights='weight' if weighted else None,
        resolution_parameter=resolution
    )
    
    # Map igraph community membership back to NetworkX nodes
    node_to_community = {}
    for vertex in ig.vs:
        node_id = vertex['node_id']
        node_to_community[node_id] = partition.membership[vertex.index]
    
    # Get unique communities
    communities_list = [
        [node for node in G.nodes() if node_to_community.get(node) == comm_idx]
        for comm_idx in range(len(partition))
    ]
    
    # Create a mapping from node to its community index
    node_to_community = {}
    for comm_idx, comm in enumerate(communities_list):
        for node in comm:
            node_to_community[node] = comm_idx
    
    # Create collapsed graph
    collapsed_G = nx.DiGraph()
    
    # For each community, create a merged node
    for comm_idx, community_nodes in enumerate(communities_list):
        if len(community_nodes) == 0:
            continue
        
        # Aggregate node data from all nodes in the community
        community_nodes_list = list(community_nodes)
        first_node = community_nodes_list[0]
        first_node_data = G.nodes[first_node]
        
        # Merge attributes
        merged_label_parts = []
        merged_func_names = set()
        is_entry_point = False
        has_non_call_links = False
        
        for node in community_nodes_list:
            node_data = G.nodes[node]
            merged_label_parts.append(node_data.get('label', f'unknown @ {node}'))
            merged_func_names.add(node_data.get('func', 'unknown'))
            is_entry_point = is_entry_point or node_data.get('entry_point', False)
            has_non_call_links = has_non_call_links or node_data.get('non_call_links', False)
        
        # Create subgraph from community nodes
        subgraph = G.subgraph(community_nodes_list).copy()
        
        # Build LLM-friendly JSON with assembly and control flow
        blocks = []
        for node in community_nodes_list:
            node_data = G.nodes[node]
            block_info = {
                'id': hex(node) if isinstance(node, int) else str(node),
                'function': node_data.get('func', 'unknown'),
                'label': node_data.get('label', f'block @ {node}'),
                'entry_point': node_data.get('entry_point', False),
                'assembly': node_data.get('instrs', [])
            }
            blocks.append(block_info)
        
        edges = []
        for src, dst, data in subgraph.edges(data=True):
            edge_info = {
                'from': hex(src) if isinstance(src, int) else str(src),
                'to': hex(dst) if isinstance(dst, int) else str(dst),
                'type': data.get('type', 'unknown'),
                'conditional': data.get('conditional', False)
            }
            edges.append(edge_info)
        
        subgraph_json_data = {
            'blocks': blocks,
            'edges': edges
        }
        json_subgraph = json.dumps(subgraph_json_data, indent=2)
        
        # Merge func names for community-level func attribute
        merged_func = ' | '.join(sorted(merged_func_names))
        
        # Get color from first function in the community
        community_color = first_node_data.get('color', 'gray')
        
        # Store only community-relevant attributes
        community_data = {
            'label': f'Community {comm_idx}',
            'func': merged_func,
            'entry_point': is_entry_point,
            'non_call_links': has_non_call_links,
            'community_size': len(community_nodes),
            'community_nodes': community_nodes_list,
            'subgraph_json': json_subgraph,
            'subgraph_object': subgraph,
            'color': community_color,
        }
        
        # Add merged node (use community index as identifier)
        collapsed_G.add_node(comm_idx, **community_data)
    
    # Add edges between communities (avoid self-loops)
    added_edges = set()
    for src_node in G.nodes():
        for dst_node in G.successors(src_node):
            src_comm = node_to_community[src_node]
            dst_comm = node_to_community[dst_node]
            
            # Skip self-loops (edges within the same community)
            if src_comm != dst_comm:
                edge_key = (src_comm, dst_comm)
                if edge_key not in added_edges:
                    # Preserve edge attributes from the original graph
                    edge_data = G.get_edge_data(src_node, dst_node)
                    collapsed_G.add_edge(src_comm, dst_comm, **edge_data)
                    added_edges.add(edge_key)
    
    return collapsed_G, communities_list

### 3. Load and Visualize the Graph with ipysigma


In [ ]:
from idadex import idaapi
from typing import Hashable
from networkx import k_core, DiGraph
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
importlib.reload(cfg)
import os


def prune_graph(og: DiGraph[Hashable]) -> DiGraph[Hashable]:
    to_return = og.copy()
    print(f"Graph loaded: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")

    # remove self loops
    to_return.remove_edges_from(nx.selfloop_edges(to_return))

    to_return = collapse_thunks(to_return)
    to_return = cfg.collapse_chains(to_return)

    # remove nodes with degree <= 1
    to_return = k_core(to_return, 2)
    entrypoint = [x for x in to_return.nodes() if to_return.nodes[x]['entry_point'] == True]
    print(f"entrypoint is {entrypoint}")
    # to_be_removed = [x for  x in G.nodes() if G.degree()[x] <= 1]
    # print(f"Number of nodes to be removed: {len(to_be_removed)}")
    # G.remove_nodes_from(to_be_removed)
    # # Basic info
    # print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")
    #
    # print(f"Number of nodes after collapsing chains: {len(G.nodes)}")
    # entrypoint = [x for x in G.nodes() if G.nodes[x]['entry_point'] == True]
    # if not entrypoint:
    #     print("No entrypoint found. Please check the graph.")
    #     raise ValueError("No entrypoint found")
    print(f"Graph pruned: {len(to_return.nodes)} nodes, {len(to_return.edges)} edges")
    return to_return

def collapse_thunks(G: nx.DiGraph):
    collapsed_G = G.copy()
    nodes_to_process = list(collapsed_G.nodes())

    for node in nodes_to_process:

        flags = collapsed_G.nodes[node].get('flags', {})
        if flags & idaapi.FUNC_THUNK:
            preds = list(collapsed_G.predecessors(node))
            succs = list(collapsed_G.successors(node))
            if len(succs) == 1:
                collapsed_G.remove_node(node)
                for pred in preds:
                    collapsed_G.add_edge(pred, succs[0])
    return collapsed_G

# Load the graph data
G = cfg.load_cfg(json_path)
if G is None:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")
    exit()

assert(G is not None)
pruned = prune_graph(G)
community_graph, communities_list = collapse_leiden(pruned, resolution=0.01, weighted=True, partition_type='CPM')
print(f"Community Graph: {len(community_graph.nodes)} nodes, {len(community_graph.edges)} edges")


def set_size(n, d) -> int:
    # Use community_size from the node data, or out_degree from community_graph
    community_size = d.get('community_size', 1)
    out_degree = community_graph.out_degree(n)
    return community_size + out_degree


In [ ]:

# Visualize with ipysigma
widget = Sigma(
    community_graph,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    node_size=set_size,
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)


In [ ]:
## needed to reload code changed on disk, this is the correct import
import src.summarize_graph as sg
importlib.reload(sg)
summaries = sg.summarize_graph(community_graph, base_url="http://192.168.1.101:8000/v1", max_concurrent=256, model="qwen3-coder-next", cache_path="../data/cache_full_func.json")
